In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

In [54]:
p_base = "../Dataset/ml-100k/u1.base"
p_test = "../Dataset/ml-100k/u1.test"
f_base = pd.read_csv(p_base,sep='\t',names=['user_id','item_id','rating','timestamp'])
f_test = pd.read_csv(p_test,sep='\t',names=['user_id','item_id','rating','timestamp'])
print(f_base.head())
print(f_test.head())

   user_id  item_id  rating  timestamp
0        1        1       5  874965758
1        1        2       3  876893171
2        1        3       4  878542960
3        1        4       3  876893119
4        1        5       3  889751712
   user_id  item_id  rating  timestamp
0        1        6       5  887431973
1        1       10       3  875693118
2        1       12       5  878542960
3        1       14       5  874965706
4        1       17       3  875073198


In [55]:
user_ratings = {} # {user_id:[rating1,rating2,...]}
item_ratings = {} # {item_id:[rating1,rating2,...]}
for (u,i,r,ts) in f_base.values:
    if u not in user_ratings:
        user_ratings[u] = []
    user_ratings[u].append(r)
    if i not in item_ratings:
        item_ratings[i] = []
    item_ratings[i].append(r)
print(user_ratings)
print(item_ratings)

{np.int64(1): [np.int64(5), np.int64(3), np.int64(4), np.int64(3), np.int64(3), np.int64(4), np.int64(1), np.int64(5), np.int64(2), np.int64(5), np.int64(5), np.int64(5), np.int64(4), np.int64(5), np.int64(1), np.int64(4), np.int64(4), np.int64(3), np.int64(4), np.int64(1), np.int64(3), np.int64(5), np.int64(2), np.int64(1), np.int64(2), np.int64(3), np.int64(3), np.int64(2), np.int64(5), np.int64(4), np.int64(5), np.int64(4), np.int64(5), np.int64(5), np.int64(4), np.int64(5), np.int64(5), np.int64(4), np.int64(5), np.int64(2), np.int64(4), np.int64(4), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(3), np.int64(5), np.int64(4), np.int64(5), np.int64(5), np.int64(2), np.int64(4), np.int64(3), np.int64(2), np.int64(2), np.int64(4), np.int64(5), np.int64(1), np.int64(5), np.int64(5), np.int64(3), np.int64(5), np.int64(3), np.int64(4), np.int64(5), np.int64(2), np.int64(5), np.int64(1), np.int64(4), np.int64(4), np.int64(3), np.int64(5), np.int64(1), np.int64(3), np.int64(3

In [56]:
user_mean = {} # {user_id:[umi]} umi: user_mean of item i
item_mean = {} # {item_id:[imi]} imi: item_mean of user i

for u,ratings in user_ratings.items():
    user_mean[u] = sum(ratings) / len(ratings)
print(user_mean)

for i,ratings in item_ratings.items():
    item_mean[i] = sum(ratings) / len(ratings)
print(item_mean)

{np.int64(1): np.float64(3.6814814814814816), np.int64(2): np.float64(3.8), np.int64(3): np.float64(3.0), np.int64(4): np.float64(4.357142857142857), np.int64(5): np.float64(2.956043956043956), np.int64(6): np.float64(3.581818181818182), np.int64(7): np.float64(3.892018779342723), np.int64(8): np.float64(3.6), np.int64(9): np.float64(4.166666666666667), np.int64(10): np.float64(4.212765957446808), np.int64(11): np.float64(3.533333333333333), np.int64(12): np.float64(4.28), np.int64(13): np.float64(3.136729222520107), np.int64(14): np.float64(4.219512195121951), np.int64(15): np.float64(3.033333333333333), np.int64(16): np.float64(4.3478260869565215), np.int64(17): np.float64(3.1578947368421053), np.int64(18): np.float64(3.9371069182389937), np.int64(19): np.float64(3.6), np.int64(20): np.float64(3.3076923076923075), np.int64(21): np.float64(2.663157894736842), np.int64(22): np.float64(3.3), np.int64(23): np.float64(3.6363636363636362), np.int64(24): np.float64(4.390243902439025), np.in

In [57]:
user_items = {} # {user_id:{item_id:[rating1,rating2,...]}}
for (u,i,r,ts) in f_base.values:
    if u not in user_items:
        user_items[u] = {}
    user_items[u][i] = r

item_users = {} # {item_id:{user_id:[rating1,rating2,...]}}
for (u,i,r,ts) in f_base.values:
    if i not in item_users:
        item_users[i] = {}
    item_users[i][u] = r

In [58]:
## 相似度度量
## 用户u和用户w之间的皮尔逊相关系数(Pearson correlation coefficient,PCC)
S_wu = {}
users = list(user_items.keys())
# users = list(df1['user_id'].unique())
# print(users)

for idx_w in range(len(users)):
    w = users[idx_w]
    items_w = user_items[w]
    r_bar_w = user_mean[w]
    for idx_u in range(idx_w + 1,len(users)):
        u = users[idx_u]
        items_u = user_items[u]
        r_bar_u = user_mean[u]
        common_items = []
        for item in items_w:
            if item in items_u:
                common_items.append(item)
        if len(common_items) < 2:
            continue

        numerator = 0.0
        for item in common_items:
            numerator += (items_u[item] - r_bar_u) * (items_w[item] - r_bar_w)

        denom_w = 0.0
        for item in common_items:
            denom_w += pow((items_w[item] - r_bar_w),2)

        denom_u = 0.0
        for item in common_items:
            denom_u += pow(items_u[item] - r_bar_u,2)

        denominator = math.sqrt(denom_w) * math.sqrt(denom_u)
        if denominator == 0.0:
            continue

        # pcc = numerator / denominator
        # significance = len(common_items) / (len(common_items) + 50)
        # S_wu[(w,u)] = pcc * significance

        S_wu[(w,u)] = numerator / denominator 


In [59]:
## 邻居选择
## 对于目标用户u，把与之相似度最高的K个人挑出来:
# 将S_wu按用户组织好：{u:{w:sim,...}}
user_sims = {}
for (w,u), sim in S_wu.items():
    if w not in user_sims:
        user_sims[w] = {}
    if u not in user_sims:
        user_sims[u] = {}
    user_sims[w][u] = sim
    user_sims[u][w] = sim
    # Pearson是对称的，因此两个方向都要存
# print(user_sims[1])
# print(user_sims[1][2])

In [60]:
K = 50
preds_ucf = []
for u, i, r, ts in f_test.values:
    # 1) 先找"评过物品 i"的所有用户（排除 u 自己）
    candidates = [w for w in user_items if i in user_items[w] and w != u]

    # 2) 在这些用户里，按与 u 的相似度取 top-K（只留 sim > 0）
    scored = [(w, user_sims[u].get(w, 0.0)) for w in candidates]
    scored = [(w, s) for (w, s) in scored if s > 0]
    scored.sort(key=lambda x: -x[1])          # 降序
    top_k = scored[:K]

    # 3) 加权平均（此时 top_k 里的人一定都评过 i）
    numerator = 0.0
    denominator = 0.0
    for w, sim in top_k:
        numerator += sim * (user_items[w][i] - user_mean[w])
        denominator += abs(sim)

    r_hat = user_mean[u] + numerator / denominator if denominator > 0 else user_mean[u]

    if r_hat < 1:
        r_hat = 1
    elif r_hat > 5:
        r_hat = 5

    preds_ucf.append(r_hat)


In [61]:
sse = 0.0
sae = 0.0
for idx, (u, i, r, ts) in enumerate(f_test.values):
    err = r - preds_ucf[idx]
    sse += err ** 2
    sae += abs(err)

print(f"UCF  RMSE = {(sse / len(preds_ucf))**0.5:.4f}")
print(f"UCF  MAE  = {sae / len(preds_ucf):.4f}")


UCF  RMSE = 0.9544
UCF  MAE  = 0.7467


In [62]:
## 相似度度量
## 调整后的物品k和物品j之间的余弦相似度(adjusted cosine similarity,ACS)

# ACS
S_kj = {}
items = list(item_users.keys())

for idx_k in range(len(items)):
    k = items[idx_k]
    users_k = item_users[k]        # {user_id: rating}
    for idx_j in range(idx_k + 1, len(items)):
        j = items[idx_j]
        users_j = item_users[j]

        common_users = []
        for user in users_k:
            if user in users_j:
                common_users.append(user)

        if len(common_users) < 2:
            continue

        numerator = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]  
            numerator += (users_k[user] - r_bar_user) * (users_j[user] - r_bar_user)

        denom_k = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]
            denom_k += (users_k[user] - r_bar_user) ** 2

        denom_j = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]
            denom_j += (users_j[user] - r_bar_user) ** 2

        denominator = math.sqrt(denom_k) * math.sqrt(denom_j)
        if denominator == 0.0:
            continue
        # pcc = numerator / denominator
        # significance = len(common_users) / (len(common_users) + 50)
        # S_kj[(k,j)] = pcc * significance
        
        S_kj[(k, j)] = numerator / denominator


In [63]:
## 邻居选择
## 对于目标物品j，把与之相似度最高的K个物品挑出来:
# 将S_kj按物品组织好：{j:{k:sim,...}}
item_sims = {}
for (k,j), sim in S_kj.items():
    if k not in item_sims:
        item_sims[k] = {}
    if j not in item_sims:
        item_sims[j] = {}
    item_sims[k][j] = sim
    item_sims[j][k] = sim
    # ACS是对称的，因此两个方向都要存


In [64]:

K = 50

preds_icf = []
for u, i, r, ts in f_test.values:
    # 1) 候选物品 = 用户 u 评过的所有物品（排除 i 自己）
    candidates = [k for k in item_users if u in item_users[k] and k != i]

    # 2) 在其中按与 i 的相似度取 top-K（只留 sim > 0）
    scored = [(k, item_sims.get(i,{}).get(k, 0.0)) for k in candidates]
    scored = [(k, s) for (k, s) in scored if s > 0]
    scored.sort(key=lambda x: -x[1])          # 相似度降序
    top_k = scored[:K]

    # 3) 中心化加权平均（与 UCF 对称）
    numerator = 0.0
    denominator = 0.0
    for k, sim in top_k:
        numerator   += sim * item_users[k][u]
        denominator += abs(sim)

    if denominator > 0:
        r_hat = numerator / denominator
    else:
        r_hat = item_mean.get(i, user_mean[u])   # 无邻居 -> 物品均值(或用户均值)兜底

    if r_hat < 1:
        r_hat = 1
    elif r_hat > 5:
        r_hat = 5

    preds_icf.append(r_hat)


In [65]:
sse = 0.0
sae = 0.0
for idx, (u, i, r, ts) in enumerate(f_test.values):
    err = r - preds_icf[idx]
    sse += err ** 2
    sae += abs(err)

print(f"ICF  RMSE = {(sse / len(preds_icf))**0.5:.4f}")
print(f"ICF  MAE  = {sae / len(preds_icf):.4f}")

ICF  RMSE = 0.9845
ICF  MAE  = 0.7753


In [66]:
preds_hybrid = []
lamda_ucf = 0.5
for r_ucf, r_icf in zip(preds_ucf, preds_icf):
    preds_hybrid.append(lamda_ucf * r_ucf + (1 - lamda_ucf) * r_icf)

# print(preds_hybrid)


In [67]:
sse = 0.0
sae = 0.0
for idx, (u, i, r, ts) in enumerate(f_test.values):
    err = r - preds_hybrid[idx]
    sse += err ** 2
    sae += abs(err)

print(f"Hybrid  RMSE = {(sse / len(preds_hybrid))**0.5:.4f}")
print(f"Hybrid  MAE  = {sae / len(preds_hybrid):.4f}")

Hybrid  RMSE = 0.9526
Hybrid  MAE  = 0.7508
